# M16 · Applications affines, graphes et variance d'un produit — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/m16_couches_affines/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/m16_couches_affines/03_notebook.ipynb`, créée par `python tools/start_chapter.py M16`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | M16.1 à M16.6, M16.10 | Les exercices papier (vérification) | ✏️ 🔗 | | |
| A | M16.12 | La matrice homogène d'une application affine · `homogeneous_matrix` | 🔨 | ★ | 15 |
|  | M16.13 | Sommer sur les chemins d'un graphe sans cycle par les puissances de sa matrice · `sum_over_paths` | 🔨 | ★★ | 20 |
|  | M16.14 | Espérance et variance d'un produit de variables indépendantes · `variance_of_product` | 🔨 | ★ | 15 |
| B | M16.15 | Prédire l'écart-type d'une somme de produits aléatoires, mesures corrélées ou non, puis simuler | 🔮 | ★ | 10 |
|  | M16.16 | Une variance multipliée d'étage en étage : simuler une chaîne d'amplificateurs bruités | 🔬 | ★★ | 20 |
|  | M16.17 | L'image d'une figure par une suite d'applications affines du plan, en coordonnées homogènes | 🎨 | ★ | 15 |
| C | M16.18 | Chasse à l'erreur : une matrice homogène écrite pour des colonnes et appliquée à des lignes, Var(XY) = Var X · Var Y, et (I − A)⁻¹ sur un graphe qui a un cycle | 🐛 | ★ | 15 |
|  | M16.19 | Un test de propriété : la matrice homogène d'une composée est le produit des matrices homogènes | 🛠️ | ★ | 15 |
|  | M16.20 | Défi : la loi du produit de deux uniformes indépendantes, sa densité et ses deux premiers moments | 🏆 | ★★★ | 25 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M16")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : une application affine dans les deux conventions et ses paramètres, des matrices homogènes, leur produit et leur inverse, des chemins comptés par les puissances de la matrice d'adjacence, un planning de tâches sans cycle, la variance d'un produit et d'une somme de produits par des signes aléatoires, des points rangés en lignes et leurs images $\mathbf{X}\mathbf{W} + \mathbf{1}\mathbf{b}^\top$.
- Écrire les trois fonctions de `mymath/affine.py` et les tester.
- Prédire puis simuler l'écart-type d'une somme de produits, suivre une variance d'étage en étage, tracer les images d'une figure, corriger des erreurs classiques, écrire un test de propriété, et relever un défi : la loi du produit de deux uniformes.

**Rappel express.** En lignes, $\mathbf{x} \mapsto \mathbf{x}\mathbf{W} + \mathbf{b}$, $\mathbf{W}$ de forme $(n, m)$, $\mathbf{b}$ de forme $(m,)$, $nm + m$ paramètres ; en colonnes, la matrice est $\mathbf{W}^\top$. $[\mathbf{x}, 1]\,\mathbf{H} = [\mathbf{x}\mathbf{W} + \mathbf{b}, 1]$ avec $\mathbf{H} = \begin{pmatrix} \mathbf{W} & \mathbf{0} \\ \mathbf{b} & 1 \end{pmatrix}$ ; $f_1$ puis $f_2$ : $\mathbf{H}_1\mathbf{H}_2$. $(\mathbf{A}^k)_{ij}$ compte les chemins de $k$ arêtes de $i$ à $j$ ; sans cycle $\iff \mathbf{A}^n = \mathbf{0}$, et $(\mathbf{I} - \mathbf{A})^{-1} = \mathbf{I} + \mathbf{A} + \cdots + \mathbf{A}^{n-1}$. $X$, $Y$ indépendantes : $\mathbb{V}(XY) = \mathbb{V}(X)\,\mathbb{V}(Y) + \mathbb{V}(X)\,\mathbb{E}[Y]^2 + \mathbb{V}(Y)\,\mathbb{E}[X]^2$ ; des $\varepsilon_i$ centrés et indépendants de tout : $\mathbb{V}\big(\sum_i\varepsilon_ix_i\big) = \sum_i\mathbb{V}(\varepsilon_i)\,\mathbb{E}[x_i^2]$ ; $v_{k+1} = g^2v_k + s^2$.

## Partie 0 · Vérifier tes exercices papier (✏️ M16.1 à M16.6, 🔗 M16.10)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- un entier, ou un nombre avec un point décimal, `-1.25`, arrondi comme le demande la question ;
- une liste `[0.2, -1]` ;
- une matrice comme la liste de ses **lignes** : `[[1, 0], [-2, 3]]` ; un nombre décimal s'y écrit avec un point, `[[0.5, 1], [0, 2]]`, ou comme un calcul, `[[1/2, 1], [0, 2]]`.

Les réponses pas encore remplies affichent ⏳.

**Ex M16.1 — Une application affine de ℝ³ dans ℝ² dans les deux conventions : formes, transposée, nombre de paramètres**

In [ ]:
# Ex M16.1: the values you found on paper
answer_M16_1a = ...  # a) a matrix (list of rows)
answer_M16_1b = ...  # b) a list of 2 numbers
answer_M16_1c = ...  # c) a list of 2 numbers
answer_M16_1d = ...  # d) a matrix (list of rows)
answer_M16_1e = ...  # e) an integer
answer_M16_1f = ...  # f) a matrix (list of rows)
answer_M16_1g = ...  # g) an integer
answer_M16_1h = ...  # h) an integer

for letter, answer in zip("abcdefgh", [answer_M16_1a, answer_M16_1b, answer_M16_1c, answer_M16_1d, answer_M16_1e, answer_M16_1f, answer_M16_1g, answer_M16_1h]):
    wbm.check(f"M16.1{letter}", answer)

**Ex M16.2 — Rotation, homothétie puis translation du plan : la matrice homogène de la composée, et celle de la réciproque**

In [ ]:
# Ex M16.2: the values you found on paper
answer_M16_2a = ...  # a) a matrix (list of rows)
answer_M16_2b = ...  # b) a matrix (list of rows)
answer_M16_2c = ...  # c) a matrix (list of rows)
answer_M16_2d = ...  # d) a matrix (list of rows)
answer_M16_2e = ...  # e) a list of 2 numbers
answer_M16_2f = ...  # f) a matrix (list of rows, decimal numbers)
answer_M16_2g = ...  # g) a list of 2 numbers

for letter, answer in zip("abcdefg", [answer_M16_2a, answer_M16_2b, answer_M16_2c, answer_M16_2d, answer_M16_2e, answer_M16_2f, answer_M16_2g]):
    wbm.check(f"M16.2{letter}", answer)

**Ex M16.3 — Des rues à sens unique : degrés, matrice d'adjacence, et chemins de longueur 2 et 3 par ses puissances**

In [ ]:
# Ex M16.3: the values you found on paper
answer_M16_3a = ...  # a) a matrix (list of rows)
answer_M16_3b = ...  # b) a list of 5 integers
answer_M16_3c = ...  # c) a list of 5 integers
answer_M16_3d = ...  # d) a matrix (list of rows)
answer_M16_3e = ...  # e) an integer
answer_M16_3f = ...  # f) an integer
answer_M16_3g = ...  # g) an integer
answer_M16_3h = ...  # h) an integer

for letter, answer in zip("abcdefgh", [answer_M16_3a, answer_M16_3b, answer_M16_3c, answer_M16_3d, answer_M16_3e, answer_M16_3f, answer_M16_3g, answer_M16_3h]):
    wbm.check(f"M16.3{letter}", answer)

**Ex M16.4 — Un planning de tâches : l'absence de cycle par les puissances de A, une numérotation qui la rend triangulaire, et le nombre de chemins entre deux tâches**

In [ ]:
# Ex M16.4: the values you found on paper
answer_M16_4a = ...  # a) a matrix (list of rows)
answer_M16_4b = ...  # b) a list of 6 integers
answer_M16_4c = ...  # c) an integer
answer_M16_4d = ...  # d) an integer
answer_M16_4e = ...  # e) a list of 6 integers
answer_M16_4f = ...  # f) an integer
answer_M16_4g = ...  # g) an integer

for letter, answer in zip("abcdefg", [answer_M16_4a, answer_M16_4b, answer_M16_4c, answer_M16_4d, answer_M16_4e, answer_M16_4f, answer_M16_4g]):
    wbm.check(f"M16.4{letter}", answer)

**Ex M16.5 — Un prix aléatoire fois une quantité indépendante : espérance et variance du produit**

In [ ]:
# Ex M16.5: the values you found on paper
answer_M16_5a = ...  # a) a number
answer_M16_5b = ...  # b) a number
answer_M16_5c = ...  # c) a number
answer_M16_5d = ...  # d) a number
answer_M16_5e = ...  # e) a number (2 decimals)
answer_M16_5f = ...  # f) a number
answer_M16_5g = ...  # g) a number (4 decimals)
answer_M16_5h = ...  # h) a number

for letter, answer in zip("abcdefgh", [answer_M16_5a, answer_M16_5b, answer_M16_5c, answer_M16_5d, answer_M16_5e, answer_M16_5f, answer_M16_5g, answer_M16_5h]):
    wbm.check(f"M16.5{letter}", answer)

**Ex M16.6 — Des signes aléatoires d'amplitudes différentes sur des mesures corrélées : la variance d'une somme de produits**

In [ ]:
# Ex M16.6: the values you found on paper
answer_M16_6a = ...  # a) a number
answer_M16_6b = ...  # b) a number
answer_M16_6c = ...  # c) a number
answer_M16_6d = ...  # d) a number
answer_M16_6e = ...  # e) a number
answer_M16_6f = ...  # f) a number
answer_M16_6g = ...  # g) a number

for letter, answer in zip("abcdefg", [answer_M16_6a, answer_M16_6b, answer_M16_6c, answer_M16_6d, answer_M16_6e, answer_M16_6f, answer_M16_6g]):
    wbm.check(f"M16.6{letter}", answer)

**Ex M16.10 — Pont : n points rangés en lignes, X W + 1bᵀ, et la diffusion de b**

In [ ]:
# Ex M16.10: the values you found on paper
answer_M16_10a = ...  # a) a list of 2 integers
answer_M16_10b = ...  # b) a matrix (list of rows)
answer_M16_10c = ...  # c) a matrix (list of rows)
answer_M16_10d = ...  # d) a matrix (list of rows)
answer_M16_10e = ...  # e) a matrix (list of rows)
answer_M16_10f = ...  # f) an integer

for letter, answer in zip("abcdef", [answer_M16_10a, answer_M16_10b, answer_M16_10c, answer_M16_10d, answer_M16_10e, answer_M16_10f]):
    wbm.check(f"M16.10{letter}", answer)

## Partie A · Ta librairie : la matrice homogène, la somme sur les chemins et la variance d'un produit

Fiche §2 à §5. Les trois fonctions de `mymath/affine.py`, leurs tests, et un usage de chacune.

In [ ]:
# Tools of the notebook (parts A to C)
import inspect
import math
import warnings

import matplotlib.pyplot as plt
import numpy as np

from wbm.nbtools import StopCheck, checked, drawn, fr, run_pytest_group, verdict

### Ex M16.12 — La matrice homogène d'une application affine · `homogeneous_matrix` 🔨 ★ ⏱️ 15 min
**Objectif :** programmer la matrice homogène d'une application affine pour des points en lignes, et s'en servir pour calculer une image.  
**Prérequis :** fiche §2 · **mymath :** `affine.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme en M03). Ouvre ta copie `mon_travail/mymath/affine.py`, lis la docstring du module puis celle de la fonction, remplace la ligne `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. NumPy est permis (`@`, `np.zeros`, `np.eye`, `np.hstack`, `np.vstack`…), mais pas `np.linalg.matrix_power`, `np.linalg.inv` ni `np.linalg.solve` dans `sum_over_paths` (les tests comparent tes résultats aux leurs, et un test vérifie que tu ne t'en sers pas). Ne modifie jamais les tableaux reçus : travaille sur une copie, `np.array(x, dtype=float)`. Les nombres se rendent en `float` de Python, les matrices en tableaux NumPy de flottants.

Écris `homogeneous_matrix(W, b)` (fiche §2), en suivant sa docstring : `ValueError` si `W` n'est pas un tableau 2-D, si `b` n'est pas 1-D ou si `len(b) != W.shape[1]` ; sinon la matrice $(n + 1, m + 1)$
$$\mathbf{H} = \begin{pmatrix} \mathbf{W} & \mathbf{0} \\ \mathbf{b} & 1 \end{pmatrix},$$
avec $\mathbf{W}$ en haut à gauche, une colonne de $n$ zéros à droite, puis la ligne $\mathbf{b}$ suivie de 1 : elle envoie $[\mathbf{x}, 1]$ sur $[\mathbf{x}\mathbf{W} + \mathbf{b}, 1]$.

Les tests tournent d'abord. La cellule d'usage calcule ensuite, avec ta fonction, la matrice homogène de l'application $\mathbf{x} \mapsto \mathbf{x}\mathbf{W} + \mathbf{b}$ de la cellule fournie ($\mathbf{W} = \begin{pmatrix} 1 & -1 \\ 0 & 2 \end{pmatrix}$, $\mathbf{b} = (3, 1)$), puis $[2, -1, 1]\,\mathbf{H}$ :
- a) l'image du point $(2, -1)$, les deux premières coordonnées de $[2, -1, 1]\,\mathbf{H}$ (vérifiée par `wbm.check`).

Dans ta copie : écris $\mathbf{H}$ à la main et refais a). Pourquoi la dernière coordonnée de $[\mathbf{x}, 1]\,\mathbf{H}$ vaut-elle 1 pour tout $\mathbf{x}$ ? Quelle est la forme de la matrice homogène d'une application de $\mathbb{R}^4$ dans $\mathbb{R}^2$, et que vaut sa dernière colonne ?

In [ ]:
W_12 = np.array([[1.0, -1.0],
                 [0.0, 2.0]])        # an affine map of the plane, x ↦ x W_12 + B_12 (points in rows)
B_12 = np.array([3.0, 1.0])
P_12 = np.array([2.0, -1.0])         # a point of the plane

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M16")  # reload your saved file

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m16_affine.py", "test_homogeneous")

**L'usage** : la matrice homogène de `W_12` et `B_12`, puis l'image de $(2, -1)$.

In [ ]:
with checked("M16.12a"):
    H_12 = np.asarray(mymath.affine.homogeneous_matrix(W_12, B_12), dtype=float)
    if H_12.shape != (3, 3):
        print(f"❌ Ex M16.12 : la matrice homogène d'une application du plan dans lui-même est (3, 3) ; "
              f"homogeneous_matrix rend la forme {H_12.shape}")
        raise StopCheck
    image_12 = np.append(P_12, 1.0) @ H_12           # [x, 1] H
    print("H =", np.round(H_12, 6).tolist(), " ; [x, 1] H =", np.round(image_12, 6).tolist())
    wbm.check("M16.12a", image_12[:2].tolist(), computed=True)
    if not math.isclose(image_12[2], 1.0):
        print("   la dernière coordonnée de [x, 1] H doit valoir 1 : la dernière colonne de H est (0, …, 0, 1)")

### Ex M16.13 — Sommer sur les chemins d'un graphe sans cycle par les puissances de sa matrice · `sum_over_paths` 🔨 ★★ ⏱️ 20 min
**Objectif :** programmer la somme sur les chemins d'un graphe sans cycle par les puissances de sa matrice, après avoir vérifié l'absence de cycle, et compter les chemins d'un quartier en grille.  
**Prérequis :** fiche §3 et §4 · **mymath :** `affine.py` · **Parcours :** C, E

Écris `sum_over_paths(weights, source, target)` (fiche §3 et §4), en suivant sa docstring : `ValueError` si `weights` n'est pas une matrice carrée, si un nœud n'est pas un entier de $[0, n - 1]$, ou si le graphe a un cycle (le message est `"the graph has a cycle"`) ; sinon la somme, sur tous les chemins de `source` à `target`, des produits des poids de leurs arêtes, $\sum_{k \ge 1}(\mathbf{A}^k)_{st}$, un `float` ($0$ quand `source == target`). Le cycle se cherche sur le **motif** des poids non nuls, `weights != 0`, par ses puissances booléennes : le graphe est sans cycle si et seulement si la $n$-ième est nulle. Pas de `np.linalg.matrix_power`, de `np.linalg.inv` ni de `np.linalg.solve` : un produit `@` par puissance.

La cellule fournie construit `GRID_13`, la matrice d'adjacence d'un quartier de $5 \times 5$ carrefours : le carrefour $(i, j)$, pour $i$ et $j$ de 0 à 4, est le nœud $5i + j$, et des rues à sens unique vont vers l'est, $(i, j) \to (i + 1, j)$, et vers le nord, $(i, j) \to (i, j + 1)$. Avec ta fonction, calcule :
- a) le nombre de chemins du coin $(0, 0)$ au coin $(4, 4)$.

Une dernière cellule ajoute à la grille une rue en travers qui revient en arrière, de $(2, 2)$ vers $(1, 2)$ : ta fonction doit alors lever `ValueError`.

Dans ta copie : retrouve a) par un dénombrement (R8) : un chemin du coin $(0, 0)$ au coin $(4, 4)$ est une suite de pas vers l'est et vers le nord ; combien de pas de chaque sorte ? Pourquoi tous ces chemins ont-ils la même longueur, et quelle puissance de `GRID_13` les compte ? Avec la rue qui revient en arrière, pourquoi la « somme sur les chemins » n'a-t-elle plus de sens ? Combien de produits `@` ta fonction fait-elle pour 25 nœuds ?

In [ ]:
SIZE_13 = 5                                    # a grid of 5 × 5 crossroads
GRID_13 = np.zeros((SIZE_13 ** 2, SIZE_13 ** 2))   # the crossroads (i, j) is the node 5 i + j
for i in range(SIZE_13):
    for j in range(SIZE_13):
        if i + 1 < SIZE_13:
            GRID_13[SIZE_13 * i + j, SIZE_13 * (i + 1) + j] = 1.0   # the one-way street to the east, (i, j) → (i + 1, j)
        if j + 1 < SIZE_13:
            GRID_13[SIZE_13 * i + j, SIZE_13 * i + j + 1] = 1.0     # the one-way street to the north, (i, j) → (i, j + 1)

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M16")  # reload your saved file

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m16_affine.py", "test_paths or test_sum_over_paths")

**L'usage** : les chemins du quartier en grille.

In [ ]:
answer_M16_13a = ...
with wbm.attempt("M16.13"):          # if you call sum_over_paths here: ⏳ while it is not written
    answer_M16_13a = ...  # a) a number: the number of paths from the corner (0, 0) to the corner (4, 4)

In [ ]:
wbm.check("M16.13a", answer_M16_13a)

with checked("M16.13"):                       # the same grid, and one street that goes back
    back_13 = GRID_13.copy()
    back_13[SIZE_13 * 2 + 2, SIZE_13 * 1 + 2] = 1.0    # the street (2, 2) → (1, 2), against the street (1, 2) → (2, 2)
    try:
        value_13 = mymath.affine.sum_over_paths(back_13, 0, SIZE_13 ** 2 - 1)
    except NotImplementedError:
        raise
    except ValueError as exc:
        print(f"✅ Ex M16.13 : avec la rue qui revient en arrière, sum_over_paths lève ValueError : {exc}")
    else:
        print(f"❌ Ex M16.13 : avec la rue qui revient en arrière, le graphe a un cycle, et sum_over_paths rend "
              f"{value_13!r} au lieu de lever ValueError(\"the graph has a cycle\")")

### Ex M16.14 — Espérance et variance d'un produit de variables indépendantes · `variance_of_product` 🔨 ★ ⏱️ 15 min
**Objectif :** programmer l'espérance et la variance d'un produit de variables indépendantes, et s'en servir pour l'incertitude d'une surface.  
**Prérequis :** fiche §5 · **mymath :** `affine.py` · **Parcours :** C

Écris `variance_of_product(mean_x, var_x, mean_y, var_y)` (fiche §5), en suivant sa docstring : `ValueError` si une variance est négative (ou si un argument n'est pas un nombre fini) ; sinon le couple $\big(\mathbb{E}[XY], \mathbb{V}(XY)\big)$ pour $X$ et $Y$ indépendantes, deux `float` :
$$\mathbb{E}[XY] = \mathbb{E}[X]\,\mathbb{E}[Y], \qquad \mathbb{V}(XY) = \mathbb{V}(X)\,\mathbb{V}(Y) + \mathbb{V}(X)\,\mathbb{E}[Y]^2 + \mathbb{V}(Y)\,\mathbb{E}[X]^2.$$

**L'usage.** Un champ rectangulaire est mesuré au décamètre : sa longueur $L$ a pour espérance 85 m et pour écart-type 0,4 m, sa largeur $\ell$ pour espérance 42 m et pour écart-type 0,3 m, les deux mesures étant indépendantes (cellule fournie). Avec ta fonction, calcule :
- a) l'écart-type de la surface $L\ell$, en m² (2 décimales).

Dans ta copie : lequel des trois termes de la variance domine, et pourquoi le produit des variances ne compte-t-il presque pas ? Retrouve a) par la forme multiplicative $1 + c_{L\ell}^2 = (1 + c_L^2)(1 + c_\ell^2)$. Que deviendrait a) si la longueur était connue exactement ?

In [ ]:
MEAN_L_14, SD_L_14 = 85.0, 0.4      # the length of the field, in m: its expectation and its standard deviation
MEAN_W_14, SD_W_14 = 42.0, 0.3      # its width, in m (the two measures are independent)

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M16")  # reload your saved file

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m16_affine.py", "test_product")

**L'usage** : l'incertitude de la surface du champ.

In [ ]:
answer_M16_14a = ...
with wbm.attempt("M16.14"):          # if you call variance_of_product here: ⏳ while it is not written
    answer_M16_14a = ...  # a) a number (2 decimals): the standard deviation of the area, in m²

In [ ]:
wbm.check("M16.14a", answer_M16_14a)

## Partie B · Des expériences et une figure

Fiche §2, §6 et §7. L'écart-type d'une somme de produits par des signes aléatoires, prédit puis simulé ; une variance multipliée d'étage en étage ; les images d'une figure par des applications affines.

### Ex M16.15 — Prédire l'écart-type d'une somme de produits aléatoires, mesures corrélées ou non, puis simuler 🔮 ★ ⏱️ 10 min
**Objectif :** prévoir l'écart-type d'une somme de mesures, corrélées ou non, avec ou sans signes aléatoires, puis le vérifier par une simulation.  
**Prérequis :** fiche §6 et §7 · **Parcours :** C, E

On additionne $n = 25$ mesures $x_1, \ldots, x_{25}$, chacune de loi $\mathcal{N}(0, 1)$, dans deux cas. **Cas 1** : les 25 mesures sont **égales**, $x_i = x$ pour tout $i$, avec $x$ de loi $\mathcal{N}(0, 1)$ (un seul tirage, recopié 25 fois). **Cas 2** : les 25 mesures sont **indépendantes**. On les multiplie aussi par des signes $\varepsilon_i = \pm 1$, chacun avec la probabilité $\frac{1}{2}$, indépendants entre eux et des mesures. Pour chaque question, choisis le numéro de la bonne réponse :
- a) l'écart-type de $\sum_i x_i$ dans le cas 1 : (1) 5 ; (2) 25 ; (3) 625 ;
- b) l'écart-type de $\sum_i x_i$ dans le cas 2 : (1) 5 ; (2) 25 ; (3) 1 ;
- c) l'écart-type de $\sum_i \varepsilon_ix_i$ dans le cas 1 : (1) 1 ; (2) 5 ; (3) 25 ;
- d) l'écart-type de $\sum_i \varepsilon_ix_i$ dans le cas 2 : (1) 1 ; (2) 5 ; (3) 25.

**Avant d'exécuter quoi que ce soit**, écris tes quatre prédictions (le **numéro du choix**), vérifie-les, puis lance l'expérience : 200 000 tirages de chaque cas (graine 4931).

In [ ]:
N_15, N_DRAWS_15 = 25, 200000        # 25 measures, 200 000 draws

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_M16_15a = ...  # a) 1, 2 or 3, written BEFORE running the experiment below
prediction_M16_15b = ...  # b) 1, 2 or 3
prediction_M16_15c = ...  # c) 1, 2 or 3
prediction_M16_15d = ...  # d) 1, 2 or 3

In [ ]:
for letter, prediction in zip("abcd", [prediction_M16_15a, prediction_M16_15b, prediction_M16_15c,
                                      prediction_M16_15d]):
    wbm.check(f"M16.15{letter}", prediction)

**L'expérience.** Elle ne s'exécute qu'une fois tes quatre prédictions écrites.

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_M16_15a, prediction_M16_15b, prediction_M16_15c, prediction_M16_15d]):
    print("⏳ Écris d'abord tes quatre prédictions, puis exécute à nouveau.")
else:
    rng_15 = np.random.default_rng(4931)
    same_15 = np.repeat(rng_15.normal(size=(N_DRAWS_15, 1)), N_15, axis=1)   # case 1: the 25 measures are equal
    independent_15 = rng_15.normal(size=(N_DRAWS_15, N_15))                  # case 2: 25 independent measures
    signs_15 = rng_15.choice([-1.0, 1.0], size=(N_DRAWS_15, N_15))           # independent random signs ±1
    for case_15, measures_15 in (("cas 1, mesures égales", same_15), ("cas 2, mesures indépendantes", independent_15)):
        plain_15 = float(np.std(measures_15.sum(axis=1)))
        signed_15 = float(np.std((signs_15 * measures_15).sum(axis=1)))
        print(f"{case_15} : écart-type de Σ xᵢ ≈ {plain_15:.3f} ; de Σ εᵢxᵢ ≈ {signed_15:.3f}".replace(".", ","))

**Après l'expérience**, dans ta copie : écris la variance de chaque somme, et dis ce qui s'additionne (les variances, les écarts-types, les covariances). Pourquoi c) et d) sont-elles égales, alors que a) et b) diffèrent tant ? Quelle hypothèse du lemme de la fiche (§6) sert dans le cas 1 ? Que deviendrait c) avec **un seul** signe $\varepsilon$ pour les 25 mesures ? Pour $n$ mesures indépendantes, comment l'écart-type de la somme et celui de la moyenne varient-ils avec $n$ ?

### Ex M16.16 — Une variance multipliée d'étage en étage : simuler une chaîne d'amplificateurs bruités 🔬 ★★ ⏱️ 20 min
**Objectif :** simuler une chaîne d'amplificateurs bruités, comparer la variance simulée à la récurrence $v_{k+1} = g^2v_k + s^2$, et voir une variance se stabiliser, croître comme $k$ ou exploser selon le gain.  
**Prérequis :** fiche §7 ; R5 · **Parcours :** C

Une chaîne de 40 amplificateurs : le signal part de $Z_0 = 0$, et chaque étage le multiplie par un gain $g$ et ajoute son propre bruit, $Z_{k+1} = gZ_k + N_k$, où les bruits $N_k$ sont de loi $\mathcal{N}(0, 1)$, indépendants entre eux et du signal. Leurs variances $v_k = \mathbb{V}(Z_k)$ vérifient $v_0 = 0$ et $v_{k+1} = g^2v_k + s^2$, avec $s^2 = 1$ (fiche §7). Ici $g$ multiplie l'amplitude $Z_k$ : la variance est multipliée par $g^2$ à chaque étage (le $\gamma$ de la fiche §7 est $g^2$). Écris :
- `simulate_chain(g, n_stages, n_chains, rng)` : le tableau $(n_{\text{chains}}, n_{\text{stages}} + 1)$ des $Z_k$ de `n_chains` chaînes indépendantes (une ligne par chaîne, une colonne par étage, $Z_0$ compris), avec des bruits tirés par `rng.normal` ;
- `variance_recurrence(g, s2, n_stages)` : le tableau NumPy $v_0 = 0, v_1, \ldots, v_{n_{\text{stages}}}$ donné par la récurrence.

Calcule ensuite, avec tes fonctions ou à la main :
- a) la limite de $v_k$ pour $g = 0{,}6$ (4 décimales) ;
- b) $v_{40}$ pour $g = 1$ (un nombre) ;
- c) $v_{40}$ pour $g = 1{,}1$ (1 décimale) ;
- d) le premier étage $k$ où $v_k$, pour $g = 0{,}6$, est à moins de 1 % de sa limite (un entier) ;
- e) vrai ou faux : pour $g = 0{,}6$, la variance empirique (`np.var`) de $Z_{40}$ sur 20 000 chaînes simulées avec un générateur de graine 4932 est à moins de 0,05 de la limite de a).

Une cellule trace ensuite, pour chacun des trois gains $g = 0{,}6$, 1 et 1,1, trente chaînes simulées et l'enveloppe $\pm 2\sqrt{v_k}$ donnée par ta récurrence, et affiche la part des 20 000 chaînes simulées qui sont dans l'enveloppe à l'étage 40.

Dans ta copie : résous $v = g^2v + 1$ pour retrouver a), et écris $v_k$ en fonction de $k$ pour $g = 0{,}6$ (R5 : une suite arithmético-géométrique). Pourquoi l'écart relatif à la limite est-il multiplié par $g^2$ à chaque étage, et que donne d) ? Que valent $v_k$ pour $g = 1$ et pour $g = 1{,}1$ ? Pour $g = 1{,}1$, de quel facteur l'enveloppe s'élargit-elle à chaque étage, au bout de quelques étages ? Pourquoi environ 95 % des chaînes restent-elles dans l'enveloppe (quelle est la loi de $Z_k$) ? Pour quels gains la variance reste-t-elle bornée ?

In [ ]:
GAINS_16 = (0.6, 1.0, 1.1)              # the gain g of each chain
N_STAGES_16, N_CHAINS_16 = 40, 20000      # 40 stages, 20 000 chains simulated for each gain

In [ ]:
def simulate_chain(g, n_stages, n_chains, rng):
    """The (n_chains, n_stages + 1) array of the Z_k of n_chains independent chains: Z_0 = 0, then
    Z_{k+1} = g Z_k + N_k, with independent noises N_k of law N(0, 1) drawn with the generator rng."""
    raise NotImplementedError("simulate_chain n'est pas encore écrite (04_indices.md si tu bloques)")


def variance_recurrence(g, s2, n_stages):
    """The NumPy array of the variances v_0 = 0, v_1, ..., v_{n_stages}, by v_{k+1} = g² v_k + s2."""
    raise NotImplementedError("variance_recurrence n'est pas encore écrite (04_indices.md si tu bloques)")


answer_M16_16a = answer_M16_16b = answer_M16_16c = answer_M16_16d = answer_M16_16e = ...
with wbm.attempt("M16.16"):          # if you call your functions here: ⏳ while they are not written
    answer_M16_16a = ...  # a) a number (4 decimals): the limit of v_k for g = 0.6
    answer_M16_16b = ...  # b) a number: v_40 for g = 1
    answer_M16_16c = ...  # c) a number (1 decimal): v_40 for g = 1.1
    answer_M16_16d = ...  # d) an integer: the first stage k where v_k (g = 0.6) is within 1 % of its limit
    answer_M16_16e = ...  # e) True or False

In [ ]:
for letter, answer in zip("abcde", [answer_M16_16a, answer_M16_16b, answer_M16_16c, answer_M16_16d,
                                     answer_M16_16e]):
    wbm.check(f"M16.16{letter}", answer)

with checked("M16.16 figure"):
    fig, axes = plt.subplots(1, len(GAINS_16), figsize=(12.0, 3.4))
    try:
        rng_fig_16 = np.random.default_rng(4933)
        stages_16 = np.arange(N_STAGES_16 + 1)
        for ax, g_16 in zip(axes, GAINS_16):
            theory_16 = np.asarray(variance_recurrence(g_16, 1.0, N_STAGES_16), dtype=float)
            chains_fig_16 = np.asarray(simulate_chain(g_16, N_STAGES_16, N_CHAINS_16, rng_fig_16), dtype=float)
            band_16 = 2 * np.sqrt(theory_16)                           # ± 2 standard deviations, from your recurrence
            inside_16 = float(np.mean(np.abs(chains_fig_16[:, -1]) <= band_16[-1]))
            ax.plot(stages_16, chains_fig_16[:30].T, color="0.65", lw=0.6)        # 30 of the simulated chains
            ax.plot(stages_16, band_16, color="C3", lw=1.8, label="± 2 √v_k (ta récurrence)")
            ax.plot(stages_16, -band_16, color="C3", lw=1.8)
            shown_16 = f"{g_16:g}".replace(".", ",")
            ax.set_title(f"g = {shown_16} : {fr(100 * inside_16, 1)} % des chaînes dans l'enveloppe à l'étage "
                         f"{N_STAGES_16}", fontsize=9)
            ax.set_xlabel("étage k")
        axes[0].set_ylabel("Z_k")
        axes[0].legend(fontsize=8, loc="upper left")
    except BaseException:
        plt.close(fig)
        raise
    plt.show()

### Ex M16.17 — L'image d'une figure par une suite d'applications affines du plan, en coordonnées homogènes 🎨 ★ ⏱️ 15 min
**Objectif :** écrire les matrices homogènes de transformations du plan et tracer les images successives d'une figure par leurs produits.  
**Prérequis :** fiche §2 ; Ex M16.12 · **Parcours :** C, E

La cellule fournie donne une flèche, `ARROW_17`, par ses 7 sommets rangés en lignes, et trois transformations du plan : $f_1$, la symétrie $(x, y) \mapsto (y, x)$ ; $f_2$, la translation de vecteur $(2, 0)$ ; $f_3$, l'homothétie de centre l'origine et de rapport 0,5. Écris :
- `transformations_17()`, qui rend la liste $[\mathbf{H}_1, \mathbf{H}_2, \mathbf{H}_3]$ de leurs matrices homogènes $3 \times 3$, pour des points en **lignes** (à la main, ou avec ta `homogeneous_matrix`, `mymath.affine.homogeneous_matrix`) ;
- `draw_images(ax, shape, matrices)`, qui dessine dans l'axe `ax` (sans en ouvrir d'autre) le polygone `shape` (un sommet par ligne), puis ses images successives par $\mathbf{H}_1$, $\mathbf{H}_1\mathbf{H}_2$, $\mathbf{H}_1\mathbf{H}_2\mathbf{H}_3$, … : passe les sommets en coordonnées homogènes $[\mathbf{x}, 1]$, multiplie-les par le produit des matrices, et garde les deux premières coordonnées. Une couleur par étape, une légende, des axes de même échelle (`ax.set_aspect("equal")`).

La cellule de vérification essaie tes trois matrices sur des points, appelle `draw_images(ax, ARROW_17, transformations_17())`, vérifie que la flèche et ses trois images sont tracées (leurs 7 sommets) dans l'axe reçu, avec une légende, puis affiche ta figure.

Dans ta copie : écris $\mathbf{H}_1\mathbf{H}_2\mathbf{H}_3$ et la formule de $(f_3 \circ f_2 \circ f_1)(x, y)$. Où va la pointe de la flèche, $(5, 2)$, à chaque étape ? Si l'on faisait l'homothétie **avant** la translation ($f_2 \circ f_3 \circ f_1$), quelle serait la dernière image, et pourquoi diffère-t-elle ? Pourquoi la dernière image a-t-elle une aire quatre fois plus petite que la flèche ?

In [ ]:
ARROW_17 = np.array([[0.0, 1.0], [3.0, 1.0], [3.0, 0.0], [5.0, 2.0],
                     [3.0, 4.0], [3.0, 3.0], [0.0, 3.0]])     # an arrow: its 7 vertices, one per row
MAPS_17 = (lambda p: p[:, ::-1],                              # f1: the symmetry (x, y) ↦ (y, x)
           lambda p: p + np.array([2.0, 0.0]),                # f2: the translation of vector (2, 0)
           lambda p: 0.5 * p)                                 # f3: the homothety of centre the origin and ratio 0.5


def drawn_shapes_17(ax):
    """The sets of points drawn in ax: the (x, y) data of its lines, the vertices of its patches, the points of its
    collections."""
    shapes = [np.column_stack([line.get_xdata(), line.get_ydata()]).astype(float) for line in ax.lines]
    shapes += [np.asarray(patch.get_xy(), dtype=float).reshape(-1, 2) for patch in ax.patches
               if hasattr(patch, "get_xy")]
    for collection in ax.collections:
        shapes += [np.asarray(path.vertices, dtype=float) for path in collection.get_paths() if len(path.vertices) > 2]
        if len(collection.get_offsets()) > 2:
            shapes.append(np.asarray(collection.get_offsets(), dtype=float))
    return shapes


def same_vertices_17(points, want):
    """True when the points drawn are exactly the vertices of want, in any order (the first one repeated or not)."""
    got = np.unique(np.round(np.asarray(points, dtype=float).reshape(-1, 2), 6), axis=0)
    want = np.unique(np.round(np.asarray(want, dtype=float), 6), axis=0)
    return got.shape == want.shape and bool(np.allclose(got, want))

In [ ]:
def transformations_17():
    """The homogeneous matrices [H1, H2, H3] (3 × 3, for points in rows) of f1, f2 and f3."""
    raise NotImplementedError("transformations_17 n'est pas encore écrite (04_indices.md si tu bloques)")


def draw_images(ax, shape, matrices):
    """In ax (and no other axes): the polygon shape (one vertex per row), then its images by H1, H1 H2, H1 H2 H3, ...
    computed in homogeneous coordinates; one colour per step, a legend."""
    raise NotImplementedError("draw_images n'est pas encore écrite (04_indices.md si tu bloques)")

In [ ]:
with checked("M16.17"):
    results_17 = []

    def report_17(ok, success, failure):
        results_17.append(bool(ok))
        print(("✅ " + success) if ok else ("❌ " + failure))

    matrices_17 = [np.asarray(H, dtype=float) for H in transformations_17()]
    if len(matrices_17) != 3:
        print(f"❌ Ex M16.17 : transformations_17 doit rendre la liste [H1, H2, H3] ; reçu {len(matrices_17)} matrice(s)")
        raise StopCheck
    probe_17 = np.array([[1.0, 2.0], [-3.0, 0.5], [0.0, -1.0]])                 # three points, in rows
    names_17 = ("H1, la symétrie (x, y) ↦ (y, x)", "H2, la translation de vecteur (2, 0)",
                "H3, l'homothétie de rapport 0,5")
    for matrix_17, map_17, name_17 in zip(matrices_17, MAPS_17, names_17):
        sends_17 = matrix_17.shape == (3, 3) and np.allclose(np.hstack([probe_17, np.ones((3, 1))]) @ matrix_17,
                                                             np.hstack([map_17(probe_17), np.ones((3, 1))]))
        report_17(sends_17, f"{name_17} envoie [x, y, 1] sur [f(x, y), 1]",
                  f"{name_17} n'envoie pas [x, y, 1] sur [f(x, y), 1] : avec des lignes, la partie linéaire en haut "
                  f"à gauche, la translation sur la dernière ligne, et la dernière colonne (0, 0, 1)")
    fig, ax = plt.subplots(figsize=(6.0, 5.0))
    try:
        opened_17 = drawn(draw_images, ax, ARROW_17, matrices_17)
    except BaseException:
        plt.close(fig)
        raise
    report_17(not opened_17, "draw_images dessine dans l'axe reçu",
              "draw_images doit dessiner dans l'axe reçu, sans ouvrir de figure (ni plt.figure ni plt.subplots)")
    shapes_17 = drawn_shapes_17(ax)
    steps_17 = [ARROW_17]
    for map_17 in MAPS_17:                              # f1, then f2 ∘ f1, then f3 ∘ f2 ∘ f1
        steps_17.append(map_17(steps_17[-1]))
    labels_17 = ("la flèche de départ", "son image par f1", "son image par f2 ∘ f1", "son image par f3 ∘ f2 ∘ f1")
    for want_17, label_17 in zip(steps_17, labels_17):
        report_17(any(same_vertices_17(s, want_17) for s in shapes_17), f"{label_17} est tracée",
                  f"je ne trouve pas {label_17} parmi tes tracés : ses 7 sommets sont les lignes de [x, 1] multipliées "
                  f"par le produit des matrices, sans la dernière coordonnée")
    entries_17 = len(ax.get_legend().get_texts()) if ax.get_legend() is not None else 0
    report_17(entries_17 >= 4, "une légende, une entrée par étape",
              f"ajoute une légende (ax.legend()), une entrée par étape (label=... dans chaque tracé) : j'en compte "
              f"{entries_17}")
    plt.show()
    verdict("M16.17", all(results_17), "les trois matrices sont justes, et la flèche et ses trois images sont tracées "
                                       "dans l'axe reçu, avec une légende.", "relis les lignes ❌ ci-dessus.")

## Partie C · Erreurs classiques, savoir-faire et défi

Fiche §1, §2, §4 et §5. Trois fonctions fausses à mesurer puis à corriger, un test de propriété de la matrice homogène, puis la loi du produit de deux uniformes.

### Ex M16.18 — Chasse à l'erreur : une matrice homogène écrite pour des colonnes et appliquée à des lignes, Var(XY) = Var X · Var Y, et (I − A)⁻¹ sur un graphe qui a un cycle 🐛 ★ ⏱️ 15 min
**Objectif :** reconnaître trois erreurs classiques qui ne lèvent aucune erreur Python (une matrice homogène dans la mauvaise convention, la variance d'un produit, une somme sur les chemins d'un graphe à cycle), mesurer leurs effets, puis les corriger.  
**Prérequis :** fiche §1, §2, §4 et §5 · **Parcours :** C, E

Une collègue a écrit trois fonctions (cellule fournie) ; leurs docstrings disent ce qu'elles doivent faire, et chacune a l'erreur que nomme le titre :
- `apply_affine(X, W, b)` doit rendre les images $\mathbf{x}\mathbf{W} + \mathbf{b}$ des points rangés en lignes dans `X`, en passant par la matrice homogène ;
- `product_variance(mean_x, var_x, mean_y, var_y)` doit rendre $\mathbb{V}(XY)$ pour $X$ et $Y$ indépendantes ;
- `paths_count(A, s, t)` doit rendre le nombre de chemins de $s$ à $t$ (deux nœuds distincts, numérotés à partir de 0) d'un graphe sans cycle, et lever une `ValueError` si le graphe a un cycle.

Aucune ne lève d'erreur sur les données de la cellule. Calcule à la main, puis vérifie en appelant les fonctions de la collègue :
- a) l'image du point `P_18` $= (1, 2)$ que rend `apply_affine(P_18, W_18, B_18)` (une liste de 2 nombres) ;
- b) la dernière coordonnée de $[1, 2, 1]\,\mathbf{H}$, avec la matrice $\mathbf{H}$ que construit `apply_affine` pour `W_18` et `B_18` (un nombre) ;
- c) ce que rend `paths_count(CYCLE_18, 0, 3)` (un nombre).

Corrige ensuite les trois fonctions : écris `apply_affine_fixed`, `product_variance_fixed` et `paths_count_fixed`, avec les mêmes arguments et les mêmes docstrings. La vérification les essaie sur des cas que tu ne vois pas : des applications de $\mathbb{R}^2$ dans $\mathbb{R}^2$, de $\mathbb{R}^3$ dans $\mathbb{R}^2$ et de $\mathbb{R}^2$ dans $\mathbb{R}^3$ ; des moyennes nulles ou non ; un graphe sans cycle à 6 nœuds, et deux graphes avec un cycle, dont un où $\mathbf{I} - \mathbf{A}$ n'est pas inversible.

In [ ]:
def apply_affine(X, W, b):
    """The images x W + b of the points of X (one point per row), computed through the homogeneous matrix of the map
    x ↦ x W + b."""
    X, W, b = np.asarray(X, dtype=float), np.asarray(W, dtype=float), np.asarray(b, dtype=float)
    n, m = W.shape
    H = np.zeros((n + 1, m + 1))
    H[:n, :m] = W                                     # the linear part, at the top left
    H[:n, m] = b                                      # the translation, in the last column
    H[n, m] = 1.0
    rows = np.hstack([X, np.ones((len(X), 1))])       # [x, 1] for every point
    return (rows @ H)[:, :-1]


def product_variance(mean_x, var_x, mean_y, var_y):
    """The variance of the product X Y of two independent variables X and Y."""
    return var_x * var_y


def paths_count(A, s, t):
    """The number of paths (of at least one edge) from s to t, two distinct nodes of a graph without cycle of
    adjacency matrix A; a ValueError if the graph has a cycle."""
    A = np.asarray(A, dtype=float)
    return float(np.linalg.inv(np.eye(len(A)) - A)[s, t])


W_18 = np.array([[2.0, 1.0],
                 [0.0, 3.0]])                     # a map of the plane, x ↦ x W_18 + B_18
B_18 = np.array([-1.0, 4.0])
P_18 = np.array([[1.0, 2.0]])                     # one point, in a row
CYCLE_18 = np.array([[0, 0, 1, 1],
                     [1, 0, 0, 0],
                     [0, 0, 0, 1],
                     [0, 1, 0, 0]])               # 0 → 2, 0 → 3, 1 → 0, 2 → 3 and 3 → 1: a graph with a cycle

In [ ]:
answer_M16_18a = ...  # a) a list of 2 numbers
answer_M16_18b = ...  # b) a number
answer_M16_18c = ...  # c) a number

In [ ]:
for letter, answer in zip("abc", [answer_M16_18a, answer_M16_18b, answer_M16_18c]):
    wbm.check(f"M16.18{letter}", answer)

**Corrige maintenant les trois fonctions.** La cellule qui suit la tienne les vérifie sur des cas que tu ne vois pas.

In [ ]:
def apply_affine_fixed(X, W, b):
    """The images x W + b of the points of X (one point per row), computed through the homogeneous matrix of the map
    x ↦ x W + b."""
    raise NotImplementedError("apply_affine_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


def product_variance_fixed(mean_x, var_x, mean_y, var_y):
    """The variance of the product X Y of two independent variables X and Y."""
    raise NotImplementedError("product_variance_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


def paths_count_fixed(A, s, t):
    """The number of paths (of at least one edge) from s to t, two distinct nodes of a graph without cycle of
    adjacency matrix A; a ValueError if the graph has a cycle."""
    raise NotImplementedError("paths_count_fixed n'est pas encore écrite (04_indices.md si tu bloques)")

In [ ]:
with checked("M16.18"):
    results_18 = []

    def report_18(ok, success, failure):
        results_18.append(bool(ok))
        print(("✅ " + success) if ok else ("❌ " + failure))

    def outcome_18(call):
        """(the value of call(), "") or (None, the error it raises); NotImplementedError goes through: ⏳."""
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", RuntimeWarning)
                return call(), ""
        except NotImplementedError:
            raise
        except Exception as exc:  # noqa: BLE001 - reported below
            return None, f"{type(exc).__name__} : {str(exc)[:120]}"

    rng_18 = np.random.default_rng(4936)                   # (A) maps of R² into R², of R³ into R², of R² into R³
    for n_18, m_18 in ((2, 2), (3, 2), (2, 3)):
        w_c18 = rng_18.integers(-4, 5, size=(n_18, m_18)).astype(float)
        b_c18 = rng_18.integers(-4, 5, size=m_18).astype(float)
        x_c18 = rng_18.integers(-3, 4, size=(4, n_18)).astype(float)
        got_18, error_18 = outcome_18(lambda: apply_affine_fixed(x_c18.copy(), w_c18.copy(), b_c18.copy()))
        good_18 = (not error_18 and got_18 is not None and np.shape(got_18) == (4, m_18)
                   and np.allclose(np.asarray(got_18, dtype=float), x_c18 @ w_c18 + b_c18))
        shown_18 = (f"ta fonction lève {error_18}" if error_18 else "reçu None" if got_18 is None
                    else f"reçu {np.round(np.asarray(got_18, dtype=float), 3).tolist()}")
        report_18(good_18, f"apply_affine_fixed : 4 points, une application de R^{n_18} dans R^{m_18}",
                  f"apply_affine_fixed, 4 points et une application de R^{n_18} dans R^{m_18} (W de forme "
                  f"({n_18}, {m_18})) : attendu les images x W + b, une ligne par point, de forme (4, {m_18}) ; "
                  f"{shown_18}")
    for args_18 in ((2.5, 0.2, -3.0, 0.4), (0.0, 2.0, 0.0, 3.0), (4.0, 0.0, -2.5, 1.2)):   # (B)
        got_18, error_18 = outcome_18(lambda: product_variance_fixed(*args_18))
        want_18 = args_18[1] * args_18[3] + args_18[1] * args_18[2] ** 2 + args_18[3] * args_18[0] ** 2
        good_18 = (not error_18 and isinstance(got_18, (int, float, np.integer, np.floating))
                   and math.isclose(float(got_18), want_18, rel_tol=1e-9))
        report_18(good_18, f"product_variance_fixed{args_18}",
                  f"product_variance_fixed{args_18} : attendu la variance de X Y pour X et Y indépendantes ; "
                  + (f"ta fonction lève {error_18}" if error_18 else f"reçu {got_18!r}"))
    rng_dag_18 = np.random.default_rng(4937)                # (C) a graph without cycle, then two graphs with a cycle
    dag_18 = np.triu((rng_dag_18.random((6, 6)) < 0.6).astype(float), 1)
    order_18 = rng_dag_18.permutation(6)
    dag_18 = dag_18[np.ix_(order_18, order_18)]
    paths_18 = sum(np.linalg.matrix_power(dag_18, k) for k in range(1, 6))    # A + A² + … + A⁵
    wrong_pairs_18 = []
    for s_18, t_18 in ((s, t) for s in range(6) for t in range(6) if s != t):
        got_18, error_18 = outcome_18(lambda: paths_count_fixed(dag_18.copy(), s_18, t_18))
        if error_18 or got_18 is None or not math.isclose(float(got_18), float(paths_18[s_18, t_18]), abs_tol=1e-9):
            wrong_pairs_18.append(f"de {s_18} à {t_18} : attendu {paths_18[s_18, t_18]:g}, "
                                  + (error_18 or f"reçu {got_18!r}"))
    report_18(not wrong_pairs_18, "paths_count_fixed : les 30 couples de nœuds d'un graphe sans cycle à 6 nœuds",
              "paths_count_fixed, sur un graphe sans cycle à 6 nœuds : " + " ; ".join(wrong_pairs_18[:2]))
    cycles_18 = {"un graphe à 5 nœuds avec un cycle, où I − A est inversible":
                     np.array([[0, 0, 1, 0, 0], [0, 0, 0, 1, 1], [1, 1, 0, 0, 1], [0, 0, 1, 0, 0], [0, 0, 0, 0, 0]]),
                 "un graphe à 4 nœuds avec le cycle 0 → 1 → 2 → 0, où I − A n'est pas inversible":
                     np.array([[0, 1, 0, 0], [0, 0, 1, 0], [1, 0, 0, 1], [0, 0, 0, 0]])}
    for label_18, cycle_18 in cycles_18.items():
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", RuntimeWarning)
                got_18 = paths_count_fixed(cycle_18.copy(), 0, len(cycle_18) - 1)
            report_18(False, "", f"paths_count_fixed, {label_18} : attendu ValueError, reçu {got_18!r}")
        except NotImplementedError:
            raise
        except Exception as exc:  # noqa: BLE001 - only a ValueError of the function itself is right
            report_18(type(exc) is ValueError, f"paths_count_fixed, {label_18} : ValueError levée ({exc})",
                      f"paths_count_fixed, {label_18} : attendu une ValueError levée par ta fonction, reçu "
                      f"{type(exc).__name__} ({str(exc)[:80]}) : vérifie l'absence de cycle AVANT de calculer")
    verdict("M16.18", all(results_18), "les trois fonctions corrigées passent tous les cas.",
            "relis les lignes ❌ ci-dessus.")

**Après la vérification**, dans ta copie : pour chaque fonction de la collègue, que calcule-t-elle exactement, et pourquoi ne lève-t-elle aucune erreur sur les données de la cellule ? Pour quelles applications `apply_affine` donnerait-elle la bonne image, et pour quelles autres planterait-elle ? Pour quelles moyennes `product_variance` a-t-elle raison ? Combien de chemins vont de 0 à 3 dans `CYCLE_18`, et que penser du nombre trouvé en c) ?

### Ex M16.19 — Un test de propriété : la matrice homogène d'une composée est le produit des matrices homogènes 🛠️ ★ ⏱️ 15 min
**Objectif :** écrire un test de propriété sur des tirages aléatoires (formes et valeurs), qui vérifie une composition sur des images de points, et s'assurer qu'il attrape des implémentations fausses.  
**Prérequis :** fiche §2 ; Ex M16.12 · **Parcours :** C

Écris `test_composition_is_a_product(homogeneous_matrix)`, une fonction de test dans le style de pytest (des `assert` avec un message qui dit le cas), qui reçoit une implémentation `homogeneous_matrix(W, b)` et vérifie la propriété du §2 : faire $f_1$ puis $f_2$, c'est multiplier $[\mathbf{x}, 1]$ par $\mathbf{H}_1\mathbf{H}_2$. Avec un générateur de graine 4935 (`np.random.default_rng(4935)`, créé dans le test), tire **au moins 20 paires** d'applications affines, $f_1 : \mathbf{x} \mapsto \mathbf{x}\mathbf{W}_1 + \mathbf{b}_1$ de $\mathbb{R}^n$ dans $\mathbb{R}^m$ et $f_2 : \mathbf{y} \mapsto \mathbf{y}\mathbf{W}_2 + \mathbf{b}_2$ de $\mathbb{R}^m$ dans $\mathbb{R}^k$, avec $n$, $m$ et $k$ tirés entre 1 et 4 et des coefficients de loi normale, puis, pour chaque paire, quelques points $\mathbf{x}$ :
- vérifie d'abord la forme de chaque matrice : $(n + 1, m + 1)$ pour $\mathbf{H}_1$, $(m + 1, k + 1)$ pour $\mathbf{H}_2$ ;
- compare $[\mathbf{x}, 1]\,\mathbf{H}_1\mathbf{H}_2$ à $[f_2(f_1(\mathbf{x})), 1]$, où $f_2(f_1(\mathbf{x}))$ est calculé directement, point par point (`np.allclose`).

La propriété se vérifie sur des **images de points** : ne calcule ni la matrice ni le vecteur de translation de la composée.

La vérification appelle ton test sur la matrice homogène de la référence, où il doit **passer**, et compte les matrices qu'il construit ; puis sur deux implémentations fausses de la cellule fournie, où il doit **échouer** (par un `assert`, pas par une autre erreur) : `homogeneous_last_column`, la translation dans la dernière colonne, comme pour des points en colonnes ; `homogeneous_corner_zero`, un 0 au lieu du 1 en bas à droite. Toutes deux ont la bonne forme.

Dans ta copie : quel `assert` attrape chacune des deux fausses matrices, et sur quel tirage ? Un test qui ne regarderait qu'**une** application à la fois, en comparant les $m$ premières coordonnées de $[\mathbf{x}, 1]\,\mathbf{H}$ à $f(\mathbf{x})$, attraperait-il `homogeneous_corner_zero` ? Et un test qui ne vérifierait que les formes ? Pourquoi tirer des $\mathbf{W}$ de toutes les formes, et des coefficients au hasard plutôt que des petits entiers choisis (que se passerait-il avec $\mathbf{b} = \mathbf{0}$) ?

In [ ]:
ref_19 = wbm.load_mymath("ref", alias="mymath_ref").affine


def homogeneous_last_column(W, b):
    """A wrong matrix: [[W, c], [0, 1]], the translation in the last column, as for column vectors (the column c
    holds b, cut or completed with zeros to the n rows of W)."""
    W, b = np.asarray(W, dtype=float), np.asarray(b, dtype=float)
    n, m = W.shape
    H = np.zeros((n + 1, m + 1))
    H[:n, :m], H[n, m] = W, 1.0
    H[:min(n, m), m] = b[:min(n, m)]
    return H


def homogeneous_corner_zero(W, b):
    """A wrong matrix: [[W, 0], [b, 0]], a 0 instead of the 1 at the bottom right."""
    W, b = np.asarray(W, dtype=float), np.asarray(b, dtype=float)
    n, m = W.shape
    H = np.zeros((n + 1, m + 1))
    H[:n, :m], H[n, :m] = W, b
    return H


WRONG_19 = {"homogeneous_last_column (la translation dans la dernière colonne)": homogeneous_last_column,
            "homogeneous_corner_zero (un 0 en bas à droite)": homogeneous_corner_zero}

In [ ]:
def test_composition_is_a_product(homogeneous_matrix):
    """[x, 1] H1 H2 = [f2(f1(x)), 1] for random affine maps f1 of R^n into R^m and f2 of R^m into R^k (seed 4935),
    checked on images of points, the shapes of H1 and H2 first."""
    raise NotImplementedError("test_composition_is_a_product n'est pas encore écrit (04_indices.md si tu bloques)")

In [ ]:
with checked("M16.19"):
    if len(inspect.signature(test_composition_is_a_product).parameters) != 1:
        raise TypeError("ton test prend un seul argument, l'implémentation à tester : "
                        "def test_composition_is_a_product(homogeneous_matrix)")
    results_19, shapes_19 = [], []

    def report_19(ok, success, failure):
        results_19.append(bool(ok))
        print(("✅ " + success) if ok else ("❌ " + failure))

    def recorded_19(W, b):
        """The reference homogeneous_matrix, which records the shape of each W."""
        shapes_19.append(tuple(np.shape(W)))
        return ref_19.homogeneous_matrix(W, b)

    try:
        test_composition_is_a_product(recorded_19)       # your test on the right matrix: ⏳ while it is not written
        why_19 = ""
    except AssertionError as exc:
        why_19 = str(exc)[:200] or "un assert sans message"
    report_19(not why_19, "ton test passe sur la bonne matrice homogène",
              f"ton test échoue sur la bonne matrice homogène (celle de la référence) : il refuse une implémentation "
              f"juste ({why_19})")
    report_19(len(shapes_19) >= 40, f"ton test construit {len(shapes_19)} matrices",
              f"ton test ne construit que {len(shapes_19)} matrice(s) : il en faut au moins 40, 20 paires "
              f"d'applications")
    report_19(any(len(s) == 2 and s[0] != s[1] for s in shapes_19), "des applications de R^n dans R^m avec n ≠ m",
              "tire aussi des applications de R^n dans R^m avec n ≠ m : les formes font partie de la propriété")
    hints_19 = {"homogeneous_last_column": "tire des translations b1 et b2 non nulles, et compare les images des "
                                           "points, pas seulement les formes",
                "homogeneous_corner_zero": "compare TOUTES les coordonnées de [x, 1] H1 H2, la dernière comprise, à "
                                           "[f2(f1(x)), 1]"}
    for label_19, wrong_19 in WRONG_19.items():
        try:
            test_composition_is_a_product(wrong_19)
            report_19(False, "", f"ton test passe sur {label_19} : il ne la voit pas ; {hints_19[wrong_19.__name__]}")
        except AssertionError:
            report_19(True, f"ton test échoue sur {label_19}", "")
        except NotImplementedError:
            raise
        except Exception as exc:  # noqa: BLE001 - a crash is not a failed assert
            report_19(False, "", f"ton test plante sur {label_19} ({type(exc).__name__} : {str(exc)[:100]}) au lieu "
                                 f"d'échouer sur un assert : vérifie d'abord la forme de chaque matrice")
    verdict("M16.19", all(results_19), "ton test accepte la bonne matrice, tire des formes variées et attrape les deux "
                                       "fausses.", "relis les lignes ❌ ci-dessus.")

### Ex M16.20 — Défi : la loi du produit de deux uniformes indépendantes, sa densité et ses deux premiers moments 🏆 ★★★ ⏱️ 25 min
**Objectif :** trouver la loi du produit de deux variables uniformes indépendantes (fonction de répartition, densité, médiane) et ses deux premiers moments, et confronter la densité à un histogramme.  
**Prérequis :** fiche §5 ; M02b · **Parcours :** C

$U$ et $V$ sont indépendantes, de loi uniforme sur $[0 ; 1]$, et $Z = UV$. Trouve :
- a) $\mathbb{E}[Z]$ (une fraction irréductible entre guillemets) ;
- b) $\mathbb{V}(Z)$ (une fraction irréductible entre guillemets) ;
- c) $P\big(Z \le \frac{1}{4}\big)$ (4 décimales) ;
- d) la médiane de $Z$, le nombre $m$ tel que $P(Z \le m) = \frac{1}{2}$ (4 décimales) ;
- e) **l'objectif chiffré** : écris `density_20(z)`, la densité de $Z$ en un point $z$ de $]0 ; 1[$ (un `float`). La vérification tire $10^6$ valeurs de $Z$ (graine 4940), en trace l'histogramme (50 classes sur $[0 ; 1]$, normalisé comme une densité), et compare ta densité à sa hauteur au centre de chaque classe de $[0{,}05 ; 0{,}95]$ : l'écart moyen doit rester **sous 0,05**.

a) et b) se calculent sans la loi de $Z$ (fiche §5). Pour c) à e), trouve d'abord la fonction de répartition $F(z) = P(Z \le z)$ pour $0 < z < 1$ : c'est l'aire de la partie $\{(u, v) : uv \le z\}$ du carré $[0 ; 1]^2$ (fais un dessin : une hyperbole coupe le carré) ; la densité est sa dérivée. Pour d), une équation à résoudre numériquement (une dichotomie, ou `scipy.optimize.brentq`).

La cellule de vérification compare tes valeurs, puis calcule l'écart de e), trace l'histogramme et ta densité, et enregistre si l'objectif est atteint.

Dans ta copie : écris $F$, puis la densité, et vérifie que son intégrale sur $]0 ; 1]$ vaut 1. Retrouve a) et b) par des intégrales avec ta densité. Pourquoi la densité tend-elle vers $+\infty$ en 0, alors que son aire reste finie ? Pourquoi la médiane est-elle si loin de l'espérance ?

In [ ]:
N_DRAWS_20 = 10 ** 6         # the draws of the check of e)

In [ ]:
def density_20(z):
    """The density of Z = U V at a point z of ]0, 1[ (a float)."""
    raise NotImplementedError("density_20 n'est pas encore écrite (04_indices.md si tu bloques)")


answer_M16_20a = ...  # a) an irreducible fraction, as text: E[Z]
answer_M16_20b = ...  # b) an irreducible fraction, as text: V(Z)
answer_M16_20c = ...  # c) a number (4 decimals): P(Z ≤ 1/4)
answer_M16_20d = ...  # d) a number (4 decimals): the median of Z

In [ ]:
for letter, answer in zip("abcd", [answer_M16_20a, answer_M16_20b, answer_M16_20c, answer_M16_20d]):
    wbm.check(f"M16.20{letter}", answer)

with checked("M16.20e"):
    rng_20 = np.random.default_rng(4940)
    draws_20 = rng_20.random(N_DRAWS_20) * rng_20.random(N_DRAWS_20)      # 10⁶ values of Z = U V
    heights_20, edges_20 = np.histogram(draws_20, bins=50, range=(0.0, 1.0), density=True)
    centers_20 = (edges_20[:-1] + edges_20[1:]) / 2
    kept_20 = (centers_20 > 0.05 - 1e-9) & (centers_20 < 0.95 + 1e-9)    # the classes of [0.05 ; 0.95]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        yours_20 = np.array([float(density_20(float(z))) for z in centers_20[kept_20]])
    gap_20 = float(np.mean(np.abs(yours_20 - heights_20[kept_20])))
    print(f"écart moyen entre ta densité et l'histogramme, sur les {int(kept_20.sum())} classes de [0,05 ; 0,95] : "
          f"{fr(gap_20, 4)} (objectif : moins de 0,05)")
    fig, ax = plt.subplots(figsize=(6.5, 3.4))
    ax.bar(centers_20, heights_20, width=edges_20[1] - edges_20[0], color="0.82", label="histogramme de 10⁶ tirages")
    ax.plot(centers_20[kept_20], yours_20, "o-", color="C3", ms=3, label="ta densité, density_20")
    ax.set_xlabel("z")
    ax.set_ylabel("densité")
    ax.legend(fontsize=8)
    plt.show()
    wbm.check("M16.20e", bool(gap_20 < 0.05), computed=True)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu écrire une application affine dans les deux conventions, avec la forme de chaque objet, et compter ses paramètres ?
2. Sais-tu composer et inverser des applications affines par leurs matrices homogènes, dans le bon ordre ?
3. Sais-tu compter des chemins par les puissances de la matrice d'adjacence, reconnaître un graphe sans cycle et sommer sur ses chemins ?
4. Sais-tu calculer l'espérance et la variance d'un produit de variables indépendantes, et la variance d'une somme de produits par des multiplicateurs centrés ?
5. Sais-tu suivre une variance multipliée d'étage en étage, et dire quand elle se stabilise, croît ou explose ?

**La suite** : le chapitre 16 de workbookIA, qui se sert de ces outils, puis M17 (jacobienne et règle de la chaîne), juste avant le chapitre 17 de workbookIA.